### 문제 1. 주문 항목 객체 만들기

카페 주문 시스템에서 “주문 항목”을 객체로 표현하려고 합니다.

아래 요구사항을 만족하는 `OrderItem` 클래스를 작성하세요.

1. 속성: `name`, `unit_price`, `qty`
2. 메소드:
    - `total_price()` : `unit_price * qty` 반환
    - `__str__()` : `"아메리카노 x2 = 6000원"` 형태로 반환
3. 실행 예시를 참고하여 출력이 나오도록 테스트 코드도 작성

In [ ]:
class OrderItem:
    def __init__(self, name, unit_price, qty):
        self.name = name
        self.unit_price = unit_price
        self.qty = qty

    def total_price(self):
        return self.unit_price * self.qty

    def __str__(self):
        return f"{self.name} x{self.qty} = {self.total_price()}원"

# 테스트 코드
item1 = OrderItem("아메리카노", 3000, 2)
item2 = OrderItem("치즈케이크", 5500, 1)

print(item1)
print(item2)
print(f"총합: {item1.total_price() + item2.total_price()}원")

### 문제 2. 캡슐화 + 검증 로직

`OrderItem`의 `unit_price`와 `qty`는 음수가 될 수 없습니다.

다음 요구사항을 반영해 확장하세요.

1. `unit_price`, `qty`는 외부에서 직접 수정되지 않게(캡슐화) 처리
2. 값을 바꾸려면 `set_unit_price()`, `set_qty()` 메소드를 통해서만 바꿀 수 있게 함
3. 음수 값이 들어오면 값을 변경하지 않고 `"가격/수량은 0 이상이어야 합니다."`를 출력

In [ ]:
class OrderItem:
    def __init__(self, name, unit_price, qty):
        self.name = name
        self._unit_price = unit_price if unit_price >= 0 else 0
        self._qty = qty if qty >= 0 else 0

    def set_unit_price(self, price):
        if price >= 0:
            self._unit_price = price
        else:
            print("가격/수량은 0 이상이어야 합니다.")

    def set_qty(self, qty):
        if qty >= 0:
            self._qty = qty
        else:
            print("가격/수량은 0 이상이어야 합니다.")

    def total_price(self):
        return self._unit_price * self._qty

    def __str__(self):
        return f"{self.name} x{self._qty} = {self.total_price()}원"

### 문제 3. 상속으로 메뉴 확장

`MenuItem`을 부모 클래스로 두고 `Coffee`, `Dessert`가 상속받도록 구성하세요.

1. `MenuItem` 공통 속성: `name`, `unit_price`
2. `Coffee`는 추가 속성: `shot` (기본 1)
3. `Dessert`는 추가 속성: `size` ("S", "M", "L" 중 하나)
4. `get_info()` 메서드:
    - Coffee: `"아메리카노(샷 2) - 3000원"`
    - Dessert: `"치즈케이크(L) - 5500원"`
5. 다형성: `items = [Coffee(...), Dessert(...)]` 같은 리스트에서 `for`로 `get_info()`를 호출해도 동일하게 동작

In [ ]:
class MenuItem:
    def __init__(self, name, unit_price):
        self.name = name
        self.unit_price = unit_price

    def get_info(self):
        pass

class Coffee(MenuItem):
    def __init__(self, name, unit_price, shot=1):
        super().__init__(name, unit_price)
        self.shot = shot

    def get_info(self):
        return f"{self.name}(샷 {self.shot}) - {self.unit_price}원"

class Dessert(MenuItem):
    def __init__(self, name, unit_price, size="M"):
        super().__init__(name, unit_price)
        self.size = size

    def get_info(self):
        return f"{self.name}({self.size}) - {self.unit_price}원"

# 다형성 테스트
items = [Coffee("아메리카노", 3000, 2), Dessert("치즈케이크", 5500, "L")]
for item in items:
    print(item.get_info())

### 문제 4. 장바구니(Cart)로 집계하기

`Cart` 클래스를 만들고 아래 기능을 구현하세요.

1. `add(item: OrderItem)` : 장바구니에 담기
2. `total()` : 장바구니 총합 반환
3. `receipt()` : 영수증 출력 (각 항목 + 총합)

In [ ]:
class Cart:
    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)

    def total(self):
        return sum(item.total_price() for item in self.items)

    def receipt(self):
        print("----- 영수증 -----")
        for item in self.items:
            print(item)
        print(f"총 합계: {self.total()}원")
        print("------------------")

# 사용 예시
my_cart = Cart()
my_cart.add(OrderItem("아메리카노", 3000, 2))
my_cart.add(OrderItem("치즈케이크", 5500, 1))
my_cart.receipt()

### 문제 5. 결제 전략(Strategy) 흉내내기

결제 수단을 객체로 분리해서 처리하세요.

1. `Payment` 추상 클래스(또는 단순 인터페이스 역할) 정의: `pay(amount)` 메소드
2. `CardPayment`, `CashPayment` 구현
    - Card: `"카드로 {amount}원 결제 완료"`
    - Cash: `"현금으로 {amount}원 결제 완료"`
3. `Cart.checkout(payment)` 호출 시 총합을 계산해 결제하고, 결제 메시지를 출력

In [ ]:
from abc import ABC, abstractmethod

# 결제 추상 클래스
class Payment(ABC):
    @abstractmethod
    def pay(self, amount):
        pass

class CardPayment(Payment):
    def pay(self, amount):
        print(f"카드로 {amount}원 결제 완료")

class CashPayment(Payment):
    def pay(self, amount):
        print(f"현금으로 {amount}원 결제 완료")

# Cart 클래스에 checkout 메서드 추가 (기존 Cart 클래스 확장)
class Cart:
    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)

    def total(self):
        return sum(item.total_price() for item in self.items)

    def checkout(self, payment_method):
        amount = self.total()
        payment_method.pay(amount)

# 실행 테스트
cart = Cart()
cart.add(OrderItem("아메리카노", 3000, 2))
cart.add(OrderItem("라떼", 4000, 1))

# 결제 수단 선택 (전략 교체)
cart.checkout(CardPayment())
cart.checkout(CashPayment())